<a href="https://colab.research.google.com/github/pranavkantgaur/training_materials/blob/master/nuclear_reactor_lec_2_hermite_curves.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 2: Hermite Curves for Nuclide Concentration Evolution
## Controlling Reactor Burnup with Parametric Curves

### Objectives:
1. Review Hermite curves and their properties
2. Apply Hermite curves to model nuclide concentration evolution
3. Control reactivity swing over burnup cycle
4. Hands-on: Design a fuel cycle for target burnup

## Hermite Curves: Quick Review

### Definition
A **Hermite curve** is defined by:
- Two endpoints: $P_0$, $P_1$
- Two tangent vectors: $T_0$, $T_1$

### Parametric Form
$$P(t) = \sum_{i=0}^{3} G_i H_i(t), \quad t \in [0, 1]$$

Where:
- $G = [P_0, P_1, T_0, T_1]^T$ is the **geometry vector**
- $H_i(t)$ are **Hermite blending functions**:

$$H_0(t) = 2t^3 - 3t^2 + 1$$
$$H_1(t) = -2t^3 + 3t^2$$
$$H_2(t) = t^3 - 2t^2 + t$$
$$H_3(t) = t^3 - t^2$$

### Key Properties
1. **Interpolation**: Curve passes through $P_0$ and $P_1$
2. **Tangent control**: $P'(0) = T_0$, $P'(1) = T_1$
3. **Local control**: Changing one point affects only that segment
4. **Smoothness**: $C^1$ continuous (can join segments smoothly)

### Why Hermite for Reactor Physics?
- **Physical constraints**: Can specify initial/final states AND rates of change
- **Smooth evolution**: No unphysical discontinuities
- **Derivative access**: Easy to compute reaction rates from concentration derivatives
- **Prediction**: Interpolate between computed time steps

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint, solve_ivp
from scipy.optimize import minimize, fsolve

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Hermite blending functions
def hermite_basis(t):
    """Compute Hermite basis functions at parameter t"""
    H0 = 2*t**3 - 3*t**2 + 1
    H1 = -2*t**3 + 3*t**2
    H2 = t**3 - 2*t**2 + t
    H3 = t**3 - t**2
    return np.array([H0, H1, H2, H3])

def hermite_basis_derivative(t):
    """Compute derivatives of Hermite basis functions"""
    dH0 = 6*t**2 - 6*t
    dH1 = -6*t**2 + 6*t
    dH2 = 3*t**2 - 4*t + 1
    dH3 = 3*t**2 - 2*t
    return np.array([dH0, dH1, dH2, dH3])

def hermite_curve(t, P0, P1, T0, T1):
    """Evaluate Hermite curve at parameter t"""
    H = hermite_basis(t)
    return H[0]*P0 + H[1]*P1 + H[2]*T0 + H[3]*T1

def hermite_curve_derivative(t, P0, P1, T0, T1):
    """Evaluate Hermite curve derivative at parameter t"""
    dH = hermite_basis_derivative(t)
    return dH[0]*P0 + dH[1]*P1 + dH[2]*T0 + dH[3]*T1

# Visualize Hermite basis functions
t = np.linspace(0, 1, 100)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for i in range(4):
    H = np.array([hermite_basis(ti)[i] for ti in t])
    plt.plot(t, H, linewidth=2, label=f'H{i}(t)')
plt.xlabel('Parameter t', fontsize=12)
plt.ylabel('Basis Function Value', fontsize=12)
plt.title('Hermite Basis Functions', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for i in range(4):
    dH = np.array([hermite_basis_derivative(ti)[i] for ti in t])
    plt.plot(t, dH, linewidth=2, label=f"H{i}'(t)")
plt.xlabel('Parameter t', fontsize=12)
plt.ylabel('Basis Function Derivative', fontsize=12)
plt.title('Hermite Basis Function Derivatives', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Example 1: Simple Hermite Curve for Concentration

Let's model U-235 concentration over 100 days using a Hermite curve.

**Given**:
- Initial concentration: $N_0$ (at t=0)
- Final concentration: $N_f$ (at t=100 days)
- Initial depletion rate: $T_0$ (from physics)
- Final depletion rate: $T_f$ (estimated/optimized)

In [ ]:
# Reactor parameters
days = 100
enrichment = 0.04

# Nuclide densities (atoms/barn-cm)
rho_U = 19.1  # g/cm^3
N_A = 6.022e23
A_U = 238
N_total = rho_U * N_A / A_U * 1e-24  # atoms/barn-cm

N0_U235 = enrichment * N_total

# Target: 3% burnup of U-235 after 100 days
burnup_fraction = 0.03
Nf_U235 = N0_U235 * (1 - burnup_fraction)

# Estimate depletion rates (atoms/barn-cm/day)
# From physics: rate = sigma_f * phi * N
phi = 1e14  # neutrons/cm^2/s
sigma_f_U235 = 585e-24  # cm^2
seconds_per_day = 86400

# Initial rate (based on initial concentration)
T0 = -sigma_f_U235 * phi * N0_U235 * seconds_per_day

# Final rate (based on final concentration)
T1 = -sigma_f_U235 * phi * Nf_U235 * seconds_per_day

print(f"Initial U-235 concentration: {N0_U235:.6e} atoms/barn-cm")
print(f"Target final concentration: {Nf_U235:.6e} atoms/barn-cm")
print(f"Initial depletion rate: {T0:.6e} atoms/barn-cm/day")
print(f"Final depletion rate: {T1:.6e} atoms/barn-cm/day")

# Create Hermite curve
t_param = np.linspace(0, 1, 1000)
t_days = t_param * days

N_U235_hermite = np.array([hermite_curve(t, N0_U235, Nf_U235, T0*days, T1*days) 
                           for t in t_param])

# Compare with exact ODE solution
def simple_depletion(N, t):
    return -sigma_f_U235 * phi * N

t_ode = np.linspace(0, days*seconds_per_day, 1000)
N_exact = odeint(simple_depletion, N0_U235, t_ode)
t_ode_days = t_ode / seconds_per_day

# Plot comparison
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(t_days, N_U235_hermite, 'b-', linewidth=2.5, label='Hermite Curve')
plt.plot(t_ode_days, N_exact, 'r--', linewidth=2, label='Exact ODE Solution', alpha=0.7)
plt.plot([0, days], [N0_U235, Nf_U235], 'go', markersize=10, label='Control Points')
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel('U-235 Concentration (atoms/barn-cm)', fontsize=12)
plt.title('U-235 Depletion: Hermite vs Exact', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Error analysis
plt.subplot(1, 2, 2)
N_exact_interp = np.interp(t_days, t_ode_days, N_exact.flatten())
error = np.abs(N_U235_hermite - N_exact_interp) / N_exact_interp * 100
plt.plot(t_days, error, 'purple', linewidth=2)
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel('Relative Error (%)', fontsize=12)
plt.title('Hermite Approximation Error', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMax relative error: {np.max(error):.4f}%")
print(f"Mean relative error: {np.mean(error):.4f}%")

## Example 2: Multi-Nuclide System with Hermite Curves

Now let's model the complete U-Pu chain:
- U-235: Direct fission
- U-238: Neutron capture → Pu-239
- Pu-239: Builds up, then fissions

We'll use **separate Hermite curves** for each nuclide!

In [ ]:
# Multi-nuclide depletion system
def multi_nuclide_depletion(N, t, phi, sigma):
    """
    N = [N_U235, N_U238, N_Pu239]
    """
    N_U235, N_U238, N_Pu239 = N
    
    dN_U235 = -sigma['U235_f'] * phi * N_U235
    dN_U238 = -sigma['U238_c'] * phi * N_U238
    dN_Pu239 = sigma['U238_c'] * phi * N_U238 - sigma['Pu239_f'] * phi * N_Pu239
    
    return [dN_U235, dN_U238, dN_Pu239]

# Cross-sections
barn = 1e-24
sigma = {
    'U235_f': 585 * barn,
    'U238_c': 2.7 * barn,
    'Pu239_f': 750 * barn
}

# Initial conditions
N0 = np.array([
    0.04 * N_total,   # U-235
    0.96 * N_total,   # U-238
    0.0               # Pu-239
])

# Solve exact ODE
phi = 1e14
t_ode = np.linspace(0, days*seconds_per_day, 500)
sol_exact = odeint(multi_nuclide_depletion, N0, t_ode, args=(phi, sigma))
t_ode_days = t_ode / seconds_per_day

# Extract solution
N_U235_exact = sol_exact[:, 0]
N_U238_exact = sol_exact[:, 1]
N_Pu239_exact = sol_exact[:, 2]

# Now fit Hermite curves through computed points
# For demonstration, we'll use beginning and end points
def fit_hermite_curve(t_data, N_data):
    """Fit Hermite curve to data using endpoints and estimated derivatives"""
    P0 = N_data[0]
    P1 = N_data[-1]
    
    # Estimate tangents using finite differences
    dt = t_data[1] - t_data[0]
    T0 = (N_data[1] - N_data[0]) / dt
    T1 = (N_data[-1] - N_data[-2]) / dt
    
    # Scale tangents by total time range
    t_range = t_data[-1] - t_data[0]
    
    return P0, P1, T0 * t_range, T1 * t_range

# Fit Hermite curves
P0_U235, P1_U235, T0_U235, T1_U235 = fit_hermite_curve(t_ode_days, N_U235_exact)
P0_U238, P1_U238, T0_U238, T1_U238 = fit_hermite_curve(t_ode_days, N_U238_exact)
P0_Pu239, P1_Pu239, T0_Pu239, T1_Pu239 = fit_hermite_curve(t_ode_days, N_Pu239_exact)

# Evaluate Hermite curves
t_hermite = np.linspace(0, 1, 1000)
t_hermite_days = t_hermite * days

N_U235_herm = np.array([hermite_curve(t, P0_U235, P1_U235, T0_U235, T1_U235) 
                        for t in t_hermite])
N_U238_herm = np.array([hermite_curve(t, P0_U238, P1_U238, T0_U238, T1_U238) 
                        for t in t_hermite])
N_Pu239_herm = np.array([hermite_curve(t, P0_Pu239, P1_Pu239, T0_Pu239, T1_Pu239) 
                         for t in t_hermite])

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# U-235
axes[0, 0].plot(t_hermite_days, N_U235_herm/N_total, 'b-', linewidth=2, label='Hermite')
axes[0, 0].plot(t_ode_days, N_U235_exact/N_total, 'r--', linewidth=1.5, label='Exact', alpha=0.7)
axes[0, 0].set_xlabel('Time (days)', fontsize=11)
axes[0, 0].set_ylabel('Fraction', fontsize=11)
axes[0, 0].set_title('U-235 Evolution', fontsize=13)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# U-238
axes[0, 1].plot(t_hermite_days, N_U238_herm/N_total, 'g-', linewidth=2, label='Hermite')
axes[0, 1].plot(t_ode_days, N_U238_exact/N_total, 'r--', linewidth=1.5, label='Exact', alpha=0.7)
axes[0, 1].set_xlabel('Time (days)', fontsize=11)
axes[0, 1].set_ylabel('Fraction', fontsize=11)
axes[0, 1].set_title('U-238 Evolution', fontsize=13)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Pu-239
axes[1, 0].plot(t_hermite_days, N_Pu239_herm/N_total*100, 'purple', linewidth=2, label='Hermite')
axes[1, 0].plot(t_ode_days, N_Pu239_exact/N_total*100, 'r--', linewidth=1.5, label='Exact', alpha=0.7)
axes[1, 0].set_xlabel('Time (days)', fontsize=11)
axes[1, 0].set_ylabel('Fraction (%)', fontsize=11)
axes[1, 0].set_title('Pu-239 Buildup', fontsize=13)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Combined view
axes[1, 1].plot(t_hermite_days, N_U235_herm/N_total, 'b-', linewidth=2, label='U-235')
axes[1, 1].plot(t_hermite_days, N_U238_herm/N_total, 'g-', linewidth=2, label='U-238')
axes[1, 1].plot(t_hermite_days, N_Pu239_herm/N_total*10, 'purple', linewidth=2, label='Pu-239 (×10)')
axes[1, 1].set_xlabel('Time (days)', fontsize=11)
axes[1, 1].set_ylabel('Normalized Fraction', fontsize=11)
axes[1, 1].set_title('All Nuclides (Hermite)', fontsize=13)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate errors
N_U235_exact_interp = np.interp(t_hermite_days, t_ode_days, N_U235_exact)
error_U235 = np.max(np.abs(N_U235_herm - N_U235_exact_interp) / N_U235_exact_interp * 100)

print(f"Max U-235 error: {error_U235:.4f}%")
print(f"\nHermite curve provides smooth interpolation between time steps!")

## Example 3: Reactivity Control with Hermite Curves

**Key Challenge**: As fuel burns, reactivity changes:
- U-235 depletion → reactivity decreases
- Fission products (Xe-135, Sm-149) → strong absorbers
- Pu-239 buildup → reactivity increases (but delayed)

**Goal**: Design fuel loading to maintain near-critical operation ($k_{eff} \approx 1.0$) over 100 days.

We'll use Hermite curves to model reactivity evolution and optimize initial conditions.

In [ ]:
# Reactivity calculation (simplified)
def calculate_keff(N_U235, N_U238, N_Pu239, phi=1e14):
    """
    Simplified k_eff calculation based on nuclide concentrations
    k_eff ~ (nu*Sigma_f) / (Sigma_a)
    """
    barn = 1e-24
    
    # Fission cross-sections
    sigma_f_U235 = 585 * barn
    sigma_f_Pu239 = 750 * barn
    
    # Absorption cross-sections
    sigma_a_U235 = 681 * barn  # fission + capture
    sigma_a_U238 = 2.7 * barn
    sigma_a_Pu239 = 1018 * barn
    
    # Neutrons per fission
    nu_U235 = 2.43
    nu_Pu239 = 2.88
    
    # Macroscopic cross-sections
    Sigma_f = sigma_f_U235 * N_U235 + sigma_f_Pu239 * N_Pu239
    Sigma_a = (sigma_a_U235 * N_U235 + sigma_a_U238 * N_U238 + 
               sigma_a_Pu239 * N_Pu239)
    
    # Simplified k_eff (ignoring geometry, leakage, etc.)
    k_inf = (nu_U235 * sigma_f_U235 * N_U235 + 
             nu_Pu239 * sigma_f_Pu239 * N_Pu239) / Sigma_a
    
    # Apply geometric factor (for finite reactor)
    k_eff = k_inf * 0.95  # ~5% leakage
    
    return k_eff

# Calculate reactivity evolution using Hermite curves
rho = np.zeros_like(t_hermite)
k_eff_evolution = np.zeros_like(t_hermite)

for i, t in enumerate(t_hermite):
    k = calculate_keff(N_U235_herm[i], N_U238_herm[i], N_Pu239_herm[i])
    k_eff_evolution[i] = k
    rho[i] = (k - 1.0) / k  # reactivity in dollars (approximately)

# Plot reactivity evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# k_eff evolution
axes[0].plot(t_hermite_days, k_eff_evolution, 'b-', linewidth=2.5)
axes[0].axhline(y=1.0, color='r', linestyle='--', linewidth=2, label='Critical (k=1.0)')
axes[0].fill_between(t_hermite_days, 0.98, 1.02, alpha=0.2, color='green', 
                      label='Acceptable Range')
axes[0].set_xlabel('Time (days)', fontsize=12)
axes[0].set_ylabel('k_eff', fontsize=12)
axes[0].set_title('Effective Multiplication Factor', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reactivity
axes[1].plot(t_hermite_days, rho*1e5, 'purple', linewidth=2.5)  # Convert to pcm
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Time (days)', fontsize=12)
axes[1].set_ylabel('Reactivity (pcm)', fontsize=12)
axes[1].set_title('Reactivity Swing', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial k_eff: {k_eff_evolution[0]:.6f}")
print(f"Final k_eff: {k_eff_evolution[-1]:.6f}")
print(f"Reactivity swing: {(rho[0] - rho[-1])*1e5:.1f} pcm")
print(f"\nNote: Negative reactivity swing means reactor becomes subcritical.")
print(f"Solution: Start with higher initial enrichment or add control!")

## Example 4: Optimizing Initial Enrichment

**Design Problem**: Find the initial enrichment such that:
1. Start critical: $k_{eff}(t=0) = 1.0$
2. Remain near-critical over 100 days
3. Achieve target burnup

We'll use Hermite curves in an optimization loop!

In [ ]:
def simulate_burnup_hermite(enrichment, days, n_segments=3):
    """
    Simulate burnup using piecewise Hermite curves
    n_segments: number of Hermite curve segments
    """
    # Initial conditions
    N0 = np.array([
        enrichment * N_total,
        (1 - enrichment) * N_total,
        0.0
    ])
    
    # Time points for segments
    t_segments = np.linspace(0, days*seconds_per_day, n_segments+1)
    
    # Solve ODE for each segment
    phi = 1e14
    all_t = []
    all_N = []
    
    for i in range(n_segments):
        t_seg = np.linspace(t_segments[i], t_segments[i+1], 100)
        if i == 0:
            N_init = N0
        else:
            N_init = sol[-1, :]
        
        sol = odeint(multi_nuclide_depletion, N_init, t_seg, args=(phi, sigma))
        all_t.append(t_seg / seconds_per_day)
        all_N.append(sol)
    
    # Combine segments
    t_full = np.concatenate(all_t)
    N_full = np.vstack(all_N)
    
    # Calculate k_eff evolution
    k_eff = np.array([calculate_keff(N[0], N[1], N[2]) for N in N_full])
    
    return t_full, N_full, k_eff

def objective_function(enrichment):
    """
    Objective: minimize deviation from k_eff = 1.0 over cycle
    """
    if enrichment < 0.02 or enrichment > 0.05:
        return 1e10  # penalty for out-of-bounds
    
    t, N, k_eff = simulate_burnup_hermite(enrichment, days)
    
    # Objective: mean squared deviation from critical
    obj = np.mean((k_eff - 1.0)**2)
    
    return obj

# Optimize
print("Optimizing initial enrichment...")
from scipy.optimize import minimize_scalar

result = minimize_scalar(objective_function, bounds=(0.02, 0.05), method='bounded')
optimal_enrichment = result.x

print(f"\nOptimal enrichment: {optimal_enrichment*100:.3f}%")

# Simulate with optimal enrichment
t_opt, N_opt, k_eff_opt = simulate_burnup_hermite(optimal_enrichment, days)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# k_eff
axes[0, 0].plot(t_opt, k_eff_opt, 'b-', linewidth=2)
axes[0, 0].axhline(y=1.0, color='r', linestyle='--', linewidth=2, label='Critical')
axes[0, 0].fill_between(t_opt, 0.99, 1.01, alpha=0.2, color='green')
axes[0, 0].set_xlabel('Time (days)', fontsize=11)
axes[0, 0].set_ylabel('k_eff', fontsize=11)
axes[0, 0].set_title(f'k_eff Evolution (Enrichment={optimal_enrichment*100:.3f}%)', fontsize=13)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Nuclide concentrations
axes[0, 1].plot(t_opt, N_opt[:, 0]/N_total, 'b-', linewidth=2, label='U-235')
axes[0, 1].plot(t_opt, N_opt[:, 1]/N_total, 'g-', linewidth=2, label='U-238')
axes[0, 1].plot(t_opt, N_opt[:, 2]/N_total*10, 'purple', linewidth=2, label='Pu-239 (×10)')
axes[0, 1].set_xlabel('Time (days)', fontsize=11)
axes[0, 1].set_ylabel('Fraction', fontsize=11)
axes[0, 1].set_title('Nuclide Evolution', fontsize=13)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Burnup
burnup = (N_opt[0, 0] - N_opt[:, 0]) / N_opt[0, 0] * 100
axes[1, 0].plot(t_opt, burnup, 'orange', linewidth=2.5)
axes[1, 0].set_xlabel('Time (days)', fontsize=11)
axes[1, 0].set_ylabel('Burnup (% initial U-235)', fontsize=11)
axes[1, 0].set_title(f'Burnup Evolution (Final={burnup[-1]:.2f}%)', fontsize=13)
axes[1, 0].grid(True, alpha=0.3)

# Reactivity
rho_opt = (k_eff_opt - 1.0) / k_eff_opt * 1e5
axes[1, 1].plot(t_opt, rho_opt, 'red', linewidth=2.5)
axes[1, 1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1, 1].set_xlabel('Time (days)', fontsize=11)
axes[1, 1].set_ylabel('Reactivity (pcm)', fontsize=11)
axes[1, 1].set_title(f'Reactivity Swing: {rho_opt[0]-rho_opt[-1]:.0f} pcm', fontsize=13)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInitial k_eff: {k_eff_opt[0]:.6f}")
print(f"Final k_eff: {k_eff_opt[-1]:.6f}")
print(f"Mean k_eff: {np.mean(k_eff_opt):.6f}")
print(f"Std dev k_eff: {np.std(k_eff_opt):.6f}")

## Using Hermite Curve Derivatives

**Key Advantage**: Hermite curves give us **analytical derivatives**!

$$\frac{dN}{dt} = \frac{dN}{d\tau} \cdot \frac{d\tau}{dt}$$

where $\tau$ is the curve parameter.

This enables:
1. **Smooth reaction rates** at any time
2. **Sensitivity analysis**: How does $k_{eff}$ change with parameters?
3. **Gradient-based optimization** (preview of Lecture 5)

In [ ]:
# Compute depletion rates from Hermite curve derivatives
t_param = np.linspace(0, 1, 1000)
dt_dparam = days  # days per unit parameter

# Derivatives
dN_U235_dt = np.array([hermite_curve_derivative(t, P0_U235, P1_U235, T0_U235, T1_U235) / dt_dparam
                       for t in t_param])
dN_Pu239_dt = np.array([hermite_curve_derivative(t, P0_Pu239, P1_Pu239, T0_Pu239, T1_Pu239) / dt_dparam
                        for t in t_param])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# U-235 concentration and rate
ax1 = axes[0]
ax2 = ax1.twinx()
line1 = ax1.plot(t_hermite_days, N_U235_herm, 'b-', linewidth=2, label='Concentration')
line2 = ax2.plot(t_hermite_days, -dN_U235_dt, 'r--', linewidth=2, label='Depletion Rate')
ax1.set_xlabel('Time (days)', fontsize=12)
ax1.set_ylabel('N (atoms/barn-cm)', fontsize=12, color='b')
ax2.set_ylabel('|dN/dt| (atoms/barn-cm/day)', fontsize=12, color='r')
ax1.set_title('U-235: Concentration and Depletion Rate', fontsize=14)
ax1.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')
ax1.grid(True, alpha=0.3)

# Pu-239 concentration and rate
ax1 = axes[1]
ax2 = ax1.twinx()
line1 = ax1.plot(t_hermite_days, N_Pu239_herm, 'purple', linewidth=2, label='Concentration')
line2 = ax2.plot(t_hermite_days, dN_Pu239_dt, 'orange', linestyle='--', linewidth=2, label='Production Rate')
ax1.set_xlabel('Time (days)', fontsize=12)
ax1.set_ylabel('N (atoms/barn-cm)', fontsize=12, color='purple')
ax2.set_ylabel('dN/dt (atoms/barn-cm/day)', fontsize=12, color='orange')
ax1.set_title('Pu-239: Concentration and Production Rate', fontsize=14)
ax1.tick_params(axis='y', labelcolor='purple')
ax2.tick_params(axis='y', labelcolor='orange')
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Hermite curves provide smooth, differentiable concentration profiles!")
print("\nThis is crucial for:")
print("  1. Accurate reaction rate calculations")
print("  2. Sensitivity/perturbation analysis")
print("  3. Gradient-based optimization (Lecture 5!)")

## Summary

### What We Learned:
1. ✅ **Hermite curves** provide smooth interpolation with derivative control
2. ✅ Applied to **nuclide concentration evolution** in reactors
3. ✅ Modeled **reactivity swing** during burnup
4. ✅ **Optimized initial enrichment** to maintain criticality
5. ✅ Accessed **analytical derivatives** for rates and sensitivity

### Key Advantages:
- **Smooth representation** of discrete depletion data
- **Physical constraints** via endpoint and tangent specification
- **Computational efficiency**: fewer time steps needed
- **Design tool**: optimize curve parameters (endpoints, tangents)

### Limitations:
- Single segment may not capture complex behavior
- Need to estimate tangent vectors carefully
- Local control only (affects entire curve)

**Next Lecture**: We'll use **Bezier curves** for spatial flux optimization, where control points provide intuitive design handles for shaping power distributions!

## Exercises

1. **Multi-Segment Hermite Curves**:
   - Divide 100 days into 4 segments
   - Fit Hermite curves to each segment
   - Ensure $C^1$ continuity at joints
   - Compare accuracy with single segment

2. **Tangent Vector Sensitivity**:
   - Vary initial tangent $T_0$ by ±20%
   - How does this affect the concentration profile?
   - What about final $k_{eff}$?

3. **Different Target Burnup**:
   - Find optimal enrichment for 5% burnup
   - What about 10% burnup?
   - Is there a maximum achievable burnup?

4. **Include Fission Products**:
   - Add Xe-135 as a 4th nuclide (strong absorber)
   - How does this affect reactivity swing?
   - Can you still maintain criticality?

5. **Gradient Verification**:
   - Compute $\frac{\partial k_{eff}}{\partial \text{enrichment}}$ numerically
   - Use Hermite curve derivatives to get analytical gradient
   - Compare the two approaches